# Forward Grid On Moving Image Review

This notebook displays separate forward coordinate-lattice grid figures for the three motion-pattern cases, each overlaid on the moving image with its own colorbar. Edit the controls in the next cell to adjust grid spacing, per-case colorbar ranges, colorbar tick/title font sizes, and output saving.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import importlib.util
import sys

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import numpy as np
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
SCRIPT = ROOT.parent / "build_motion_pattern_figure.py"

spec = importlib.util.spec_from_file_location("motion_pattern_builder", SCRIPT)
mp = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = mp
spec.loader.exec_module(mp)
mp.self_check_forward_grid_helpers()

# Data/build controls
DEVICE_NAME = "cuda" if mp.torch.cuda.is_available() else "cpu"
GRID_STEP = 4
VECTOR_STEP = 12  # kept only because the shared case builder expects it

# Colorbar controls. By default each case uses its own robust max amplitude.
DEFAULT_COLORBAR_VMIN_MM = 0.0
DEFAULT_COLORBAR_VMAX_MM = None  # set to a number to use one shared upper bound for all cases
COLORBAR_RANGE_MM_BY_CASE = {
    "oasis_rank009": (0.0, 8.0),
    # "lungct_case01": (0.0, 10.0),
    "lungct_case08": (0.0, 25.0),
}
DEFAULT_COLORBAR_TICKS = None  # e.g. [0, 5, 10, 15, 20, 25]
COLORBAR_TICKS_BY_CASE = {
    # "oasis_rank009": [0, 2, 4, 6, 8],
}
COLORBAR_TITLE = "Motion amplitude (mm)"
COLORBAR_TITLE_FONTSIZE = 12
COLORBAR_TICK_FONTSIZE = 10

# Figure typography and style controls
FIGURE_TITLE = None  # set to a string to add an extra title above each case
FIGURE_TITLE_FONTSIZE = 16
PANEL_TITLE_FONTSIZE = 13
GRID_LINEWIDTH = 0.45
GRID_ALPHA = 0.95
CMAP = "viridis"
FIGSIZE_PER_CASE = (5.8, 5.0)

# Save controls
SAVE_FIGURE = True
OUTPUT_STEM_PREFIX = "forward_grid_moving_overlay"


In [ ]:
args = SimpleNamespace(
    output_dir=ROOT,
    grid_step=GRID_STEP,
    vector_step=VECTOR_STEP,
    device=DEVICE_NAME,
    oasis_root=mp.oasis_common.DEFAULT_OASIS_ROOT,
    oasis_pair_file=mp.oasis_common.DEFAULT_PAIR_FILE,
    sitreg_root=mp.DEFAULT_SITREG_ROOT,
    lungct_ptvprep_root=mp.lungct_common.DEFAULT_PTVPREP_ROOT,
    ptvreg_root=mp.DEFAULT_PTVREG_ROOT,
    overwrite=False,
)

device = mp.device_from_arg(DEVICE_NAME)
cases = [
    mp.build_oasis_case(args, device),
    mp.build_lungct_case(args, 1, device),
    mp.build_lungct_case(args, 8, device),
]

def colorbar_range_for_case(case):
    case_key = case.manifest["case_key"]
    override = COLORBAR_RANGE_MM_BY_CASE.get(case_key)
    if override is None:
        vmin = DEFAULT_COLORBAR_VMIN_MM
        vmax = case.forward_grid.color_max_mm if DEFAULT_COLORBAR_VMAX_MM is None else DEFAULT_COLORBAR_VMAX_MM
    else:
        vmin, vmax = override
        vmin = DEFAULT_COLORBAR_VMIN_MM if vmin is None else vmin
        vmax = case.forward_grid.color_max_mm if vmax is None else vmax
    vmin = float(vmin)
    vmax = float(vmax)
    if vmax <= vmin:
        raise ValueError(f"Colorbar max must be greater than min for {case_key}: {vmax} <= {vmin}")
    return vmin, vmax


def colorbar_ticks_for_case(case):
    return COLORBAR_TICKS_BY_CASE.get(case.manifest["case_key"], DEFAULT_COLORBAR_TICKS)


colorbar_ranges = {case.manifest["case_key"]: colorbar_range_for_case(case) for case in cases}

summary_lines = [
    f"Device: `{device}`",
    f"Grid step: `{GRID_STEP}` voxels",
    "Colorbar range: per-case auto robust max unless overridden",
]
for case in cases:
    summary_lines.append(
        f"{case.manifest['case_key']}: colorbar `{colorbar_ranges[case.manifest['case_key']][0]:g}` "
        f"to `{colorbar_ranges[case.manifest['case_key']][1]:g}` mm, "
        f"forward-grid robust max `{case.forward_grid.color_max_mm:.3f}` mm, sample count `{case.forward_grid.sample_count}`"
    )
display(Markdown("  \n".join(summary_lines)))


In [ ]:
def draw_forward_grid_on_moving(ax, case, norm, colorbar_vmin, colorbar_vmax):
    ax.imshow(case.moving_slice, cmap="gray", vmin=0.0, vmax=1.0, origin="lower", aspect=case.aspect)
    segments, values = mp.forward_grid_segments(case.forward_grid)
    line_collection = LineCollection(
        segments,
        cmap=CMAP,
        norm=norm,
        linewidths=GRID_LINEWIDTH,
        alpha=GRID_ALPHA,
    )
    line_collection.set_array(np.clip(values, colorbar_vmin, colorbar_vmax))
    ax.add_collection(line_collection)
    mp.set_display_axis(ax, case.forward_grid.display_shape, case.aspect)
    ax.set_title(case.row_label.replace("\n", " "), fontsize=PANEL_TITLE_FONTSIZE)
    return line_collection

def output_stem_for_case(case):
    return f"{OUTPUT_STEM_PREFIX}_{case.manifest['case_key']}"


saved_outputs = []
for case in cases:
    case_key = case.manifest["case_key"]
    colorbar_vmin, colorbar_vmax = colorbar_ranges[case_key]
    norm = mcolors.Normalize(vmin=colorbar_vmin, vmax=colorbar_vmax)

    fig, ax = plt.subplots(1, 1, figsize=FIGSIZE_PER_CASE, constrained_layout=True)
    mappable = draw_forward_grid_on_moving(ax, case, norm, colorbar_vmin, colorbar_vmax)
    if FIGURE_TITLE:
        fig.suptitle(f"{FIGURE_TITLE}: {case.row_label.replace(chr(10), ' ')}", fontsize=FIGURE_TITLE_FONTSIZE)

    colorbar = fig.colorbar(mappable, ax=ax, fraction=0.046, pad=0.03)
    colorbar.set_label(COLORBAR_TITLE, fontsize=COLORBAR_TITLE_FONTSIZE)
    ticks = colorbar_ticks_for_case(case)
    if ticks is not None:
        colorbar.set_ticks(ticks)
    colorbar.ax.tick_params(labelsize=COLORBAR_TICK_FONTSIZE)

    if SAVE_FIGURE:
        stem = output_stem_for_case(case)
        png_path = ROOT / f"{stem}.png"
        pdf_path = ROOT / f"{stem}.pdf"
        fig.savefig(png_path, dpi=300)
        fig.savefig(pdf_path)
        saved_outputs.append((png_path, pdf_path))
        display(Markdown(f"Saved `{png_path.name}` and `{pdf_path.name}`."))
        display(Image(filename=str(png_path)))
    else:
        display(fig)

    plt.close(fig)

if SAVE_FIGURE:
    display(Markdown("Generated separate forward-grid moving-overlay figures: " + ", ".join(f"`{p.name}`" for p, _ in saved_outputs)))
